# Week 4 — Transfer Learning & Image Segmentation
## Stand on the Shoulders of Giants

**IIT Madras · Wadhwani School of AI**

---

**Session Plan (60 min):**
1. Transfer Learning (~18 min) — EfficientNet-B0 on food images, freeze + replace head
2. U-Net Segmentation (~17 min) — encoder-decoder with skip connections on Oxford Pets
3. Q&A (~10 min)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, models
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time, os, zipfile, requests

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")

---
# Part 1: Transfer Learning (~18 min)

Instead of training a CNN from scratch (week 3), we take a model already trained on 1M+ ImageNet images and adapt it to our task. Three steps: load pretrained → freeze features → replace classifier.

### 1.1 Download the FoodVision Mini Dataset

3 classes — pizza, steak, sushi. Small dataset (~750 images total) where transfer learning really shines.

In [ ]:
# Download FoodVision Mini
data_path = Path('data')
image_path = data_path / 'pizza_steak_sushi'

if not image_path.exists():
    data_path.mkdir(parents=True, exist_ok=True)
    url = 'https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip'
    zip_path = data_path / 'pizza_steak_sushi.zip'

    print('Downloading FoodVision Mini...')
    r = requests.get(url)
    with open(zip_path, 'wb') as f:
        f.write(r.content)

    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(data_path)
    print('Done!')

train_dir = image_path / 'train'
test_dir = image_path / 'test'

class_names = sorted(os.listdir(train_dir))
print(f"Classes: {class_names}")
for cls in class_names:
    n_train = len(os.listdir(train_dir / cls))
    n_test = len(os.listdir(test_dir / cls))
    print(f"  {cls}: {n_train} train, {n_test} test")

### 1.2 Load Pretrained EfficientNet-B0

EfficientNet-B0 was trained on ImageNet (1M+ images, 1000 classes). It already knows edges, textures, shapes, and objects. We're borrowing all of that knowledge.

In [ ]:
# Load pretrained model
weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights).to(device)

# What does the model look like?
print(f"Model structure (abbreviated):")
print(f"  model.features  → {len(model.features)} blocks (the CNN feature extractor)")
print(f"  model.avgpool   → adaptive average pooling")
print(f"  model.classifier → {model.classifier}")

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"Currently outputs: {model.classifier[1].out_features} classes (ImageNet)")
print(f"We need: {len(class_names)} classes (pizza, steak, sushi)")

### 1.3 Freeze Features, Replace Classifier

Two key lines: `requires_grad = False` on all feature layers (freeze them), then replace the classifier head with a new one that outputs 3 classes.

In [ ]:
# Step 1: FREEZE all feature extractor layers
for param in model.features.parameters():
    param.requires_grad = False

# Step 2: REPLACE the classifier head
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280, out_features=len(class_names))
).to(device)

# Count trainable vs frozen parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)

print(f"Frozen parameters:    {frozen:,}  (feature extractor — don't touch)")
print(f"Trainable parameters: {trainable:,}  (new classifier only)")
print(f"Training {trainable/frozen*100:.2f}% of the model")
print(f"\nNew classifier: {model.classifier}")

### 1.4 Data Transforms and Loaders

The pretrained model expects images preprocessed the same way as its original training data. We use `weights.transforms()` to get the exact transforms automatically.

In [ ]:
# Get the transforms used during pretraining
auto_transforms = weights.transforms()
print(f"Auto transforms: {auto_transforms}")

# Create datasets and dataloaders
train_data = datasets.ImageFolder(train_dir, transform=auto_transforms)
test_data = datasets.ImageFolder(test_dir, transform=auto_transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"\nTrain: {len(train_data)} images")
print(f"Test:  {len(test_data)} images")
print(f"Classes: {train_data.classes}")

### 1.5 Train — Same 5-Step Loop

Exact same training loop as weeks 1–3. The only difference: most of the model is frozen, so backprop only updates the classifier head. Training is fast.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

NUM_EPOCHS = 5
train_losses, test_losses, test_accs = [], [], []

start = time.time()

for epoch in range(NUM_EPOCHS):
    # ===== TRAIN =====
    model.train()
    running_loss = 0.0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)

        pred = model(X)              # 1. Forward
        loss = loss_fn(pred, y)      # 2. Loss
        optimizer.zero_grad()        # 3. Zero grad
        loss.backward()              # 4. Backward
        optimizer.step()             # 5. Update

        running_loss += loss.item()
    train_losses.append(running_loss / len(train_loader))

    # ===== TEST =====
    model.eval()
    test_loss, correct, total = 0.0, 0, 0
    with torch.inference_mode():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()
            total += y.size(0)

    test_losses.append(test_loss / len(test_loader))
    test_accs.append(correct / total)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {train_losses[-1]:.4f} | "
          f"Test Acc: {test_accs[-1]:.1%}")

elapsed = time.time() - start
print(f"\nDone in {elapsed:.1f}s")
print(f"Final test accuracy: {test_accs[-1]:.1%} — with only 5 epochs on ~750 images!")

### 1.6 Results and Sample Predictions

Let's see the training curves and a few predictions on test images.

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
epochs_range = range(1, NUM_EPOCHS + 1)

axes[0].plot(epochs_range, train_losses, 'o-', label='Train', color='#7C6AEF', linewidth=2)
axes[0].plot(epochs_range, test_losses, 's-', label='Test', color='#FB923C', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, [a*100 for a in test_accs], 's-', color='#4ADE80', linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Test Accuracy', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Sample predictions
from PIL import Image
import random

model.eval()
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

test_images = list(test_dir.rglob('*.jpg'))
samples = random.sample(test_images, 10)

for ax, img_path in zip(axes.flat, samples):
    # Load and display original
    img_pil = Image.open(img_path)
    ax.imshow(img_pil)

    # Predict
    img_tensor = auto_transforms(img_pil).unsqueeze(0).to(device)
    with torch.inference_mode():
        logits = model(img_tensor)
        probs = F.softmax(logits, dim=1)
        conf, pred_idx = probs.max(1)

    pred_label = class_names[pred_idx.item()]
    true_label = img_path.parent.name
    correct = pred_label == true_label
    color = '#4ADE80' if correct else '#EF4444'

    ax.set_title(f'{pred_label} ({conf.item():.0%})', fontsize=10,
                 color=color, fontweight='bold')
    ax.axis('off')

fig.suptitle('Transfer Learning Predictions (green=correct, red=wrong)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"High accuracy with tiny dataset + 5 epochs.")
print(f"Training from scratch on 750 images would overfit badly.")

---
# Part 2: U-Net Segmentation (~17 min)

New task: instead of classifying the whole image, predict a class for **every pixel**. The output is a mask the same size as the input. We'll use U-Net — an encoder-decoder CNN with skip connections.

### 2.1 Load Oxford Pets with Segmentation Masks

Oxford-IIIT Pet dataset includes segmentation masks (trimaps). We convert to binary: pet (foreground) vs background.

In [ ]:
IMG_SIZE = 128  # Small for fast training in a live demo

class OxfordPetsSegmentation(Dataset):
    """Oxford Pets with binary segmentation masks."""
    def __init__(self, split='trainval', img_size=128):
        self.dataset = torchvision.datasets.OxfordIIITPet(
            root='./data', split=split, download=True,
            target_types='segmentation'
        )
        self.img_size = img_size
        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize((img_size, img_size), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.PILToTensor()
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, mask = self.dataset[idx]
        img = self.img_transform(img)
        mask = self.mask_transform(mask).float()
        # Trimap: 1=foreground, 2=background, 3=boundary
        # Convert to binary: foreground (1) vs everything else (0)
        mask = (mask == 1).float().squeeze(0)  # H×W binary mask
        return img, mask

train_seg = OxfordPetsSegmentation(split='trainval', img_size=IMG_SIZE)
test_seg = OxfordPetsSegmentation(split='test', img_size=IMG_SIZE)

seg_train_loader = DataLoader(train_seg, batch_size=16, shuffle=True)
seg_test_loader = DataLoader(test_seg, batch_size=16, shuffle=False)

print(f"Train: {len(train_seg)} images")
print(f"Test:  {len(test_seg)} images")

# Show a sample
img, mask = train_seg[0]
print(f"Image shape: {img.shape}  (C×H×W)")
print(f"Mask shape:  {mask.shape}  (H×W, binary)")

In [ ]:
# Visualize a few image-mask pairs
fig, axes = plt.subplots(2, 5, figsize=(14, 5.5))

for i in range(5):
    img, mask = train_seg[i * 50]
    # Unnormalize for display
    display = img.permute(1, 2, 0).numpy()
    display = display * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
    display = np.clip(display, 0, 1)

    axes[0, i].imshow(display)
    axes[0, i].set_title('Image', fontsize=10)
    axes[0, i].axis('off')

    axes[1, i].imshow(mask.numpy(), cmap='gray')
    axes[1, i].set_title('Mask (pet=white)', fontsize=10)
    axes[1, i].axis('off')

fig.suptitle('Oxford Pets — Image and Segmentation Mask Pairs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.2 Build the U-Net

The architecture has three parts: an **encoder** (compress spatial, same as week 3 CNN), a **decoder** (expand back using ConvTranspose2d), and **skip connections** (concatenate encoder features to decoder for detail recovery).

In [ ]:
def double_conv(in_ch, out_ch):
    """Two Conv-BN-ReLU blocks in sequence. The basic building block of U-Net."""
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True)
    )


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        # ENCODER (downsampling) — same idea as week 3 CNN
        self.enc1 = double_conv(in_channels, 64)
        self.enc2 = double_conv(64, 128)
        self.enc3 = double_conv(128, 256)
        self.enc4 = double_conv(256, 512)
        self.pool = nn.MaxPool2d(2, 2)

        # BOTTLENECK
        self.bottleneck = double_conv(512, 1024)

        # DECODER (upsampling) — NEW: ConvTranspose2d reverses MaxPool
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = double_conv(1024, 512)   # 1024 because of skip concat

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = double_conv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = double_conv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = double_conv(128, 64)

        # Final 1×1 conv to get the mask
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)                          # → 64
        e2 = self.enc2(self.pool(e1))               # → 128
        e3 = self.enc3(self.pool(e2))               # → 256
        e4 = self.enc4(self.pool(e3))               # → 512

        # Bottleneck
        b = self.bottleneck(self.pool(e4))           # → 1024

        # Decoder + skip connections (concatenate encoder features)
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))   # 512+512=1024→512
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))  # 256+256=512→256
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))  # 128+128=256→128
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))  # 64+64=128→64

        return self.final(d1)  # → 1 channel (binary mask logits)

unet = UNet(in_channels=3, out_channels=1).to(device)
unet_params = sum(p.numel() for p in unet.parameters())
print(f"U-Net parameters: {unet_params:,}")

# Verify shapes
test_in = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
test_out = unet(test_in)
print(f"Shape check: {test_in.shape} → {test_out.shape}")
print(f"Input: 3-channel RGB image. Output: 1-channel mask (same H×W).")

### 2.3 Train the U-Net

Same 5-step loop. Loss is `BCEWithLogitsLoss` — per-pixel binary classification (pet or not). We also track Dice score, which measures overlap between predicted and actual masks.

In [ ]:
def dice_score(pred, target, threshold=0.5, eps=1e-7):
    """Dice coefficient — overlap between prediction and ground truth."""
    pred_binary = (torch.sigmoid(pred) > threshold).float()
    intersection = (pred_binary * target).sum()
    return (2.0 * intersection + eps) / (pred_binary.sum() + target.sum() + eps)


seg_loss_fn = nn.BCEWithLogitsLoss()
seg_optimizer = optim.Adam(unet.parameters(), lr=1e-3)

SEG_EPOCHS = 5
seg_train_losses, seg_test_losses, seg_dices = [], [], []

start = time.time()

for epoch in range(SEG_EPOCHS):
    # ===== TRAIN =====
    unet.train()
    running_loss = 0.0
    for imgs, masks in seg_train_loader:
        imgs, masks = imgs.to(device), masks.to(device)

        pred = unet(imgs).squeeze(1)         # 1. Forward (squeeze channel dim)
        loss = seg_loss_fn(pred, masks)       # 2. Loss (per-pixel BCE)
        seg_optimizer.zero_grad()             # 3. Zero grad
        loss.backward()                       # 4. Backward
        seg_optimizer.step()                  # 5. Update

        running_loss += loss.item()
    seg_train_losses.append(running_loss / len(seg_train_loader))

    # ===== TEST =====
    unet.eval()
    test_loss, total_dice = 0.0, 0.0
    with torch.inference_mode():
        for imgs, masks in seg_test_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            pred = unet(imgs).squeeze(1)
            test_loss += seg_loss_fn(pred, masks).item()
            total_dice += dice_score(pred, masks).item()

    seg_test_losses.append(test_loss / len(seg_test_loader))
    seg_dices.append(total_dice / len(seg_test_loader))

    print(f"Epoch {epoch+1}/{SEG_EPOCHS} | "
          f"Train Loss: {seg_train_losses[-1]:.4f} | "
          f"Test Loss: {seg_test_losses[-1]:.4f} | "
          f"Dice: {seg_dices[-1]:.3f}")

elapsed = time.time() - start
print(f"\nDone in {elapsed:.1f}s")
print(f"Dice score: {seg_dices[-1]:.3f} (1.0 = perfect overlap)")

### 2.4 Visualize Predictions vs Ground Truth

The real payoff — seeing the model's predicted masks overlaid on the original images.

In [ ]:
unet.eval()
fig, axes = plt.subplots(3, 5, figsize=(15, 9))

indices = [0, 10, 25, 50, 80]

for col, idx in enumerate(indices):
    img, mask = test_seg[idx]

    # Predict
    with torch.inference_mode():
        pred = unet(img.unsqueeze(0).to(device)).squeeze()
        pred_mask = (torch.sigmoid(pred) > 0.5).float().cpu().numpy()

    # Unnormalize image
    display = img.permute(1, 2, 0).numpy()
    display = display * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
    display = np.clip(display, 0, 1)

    # Row 1: Original image
    axes[0, col].imshow(display)
    axes[0, col].set_title('Input' if col == 0 else '', fontsize=10)
    axes[0, col].axis('off')

    # Row 2: Ground truth mask
    axes[1, col].imshow(mask.numpy(), cmap='gray')
    axes[1, col].set_title('Ground Truth' if col == 0 else '', fontsize=10)
    axes[1, col].axis('off')

    # Row 3: Predicted mask
    axes[2, col].imshow(pred_mask, cmap='gray')
    d = dice_score(torch.tensor(pred_mask), mask, threshold=0.0)
    axes[2, col].set_title(f'Dice: {d:.2f}' if col > 0 else f'Predicted (Dice: {d:.2f})', fontsize=10)
    axes[2, col].axis('off')

    if col == 0:
        axes[0, col].set_ylabel('Image', fontsize=11, fontweight='bold')
        axes[1, col].set_ylabel('Truth', fontsize=11, fontweight='bold')
        axes[2, col].set_ylabel('Predicted', fontsize=11, fontweight='bold')

fig.suptitle('U-Net Predictions — Pet Segmentation', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("The model learns to outline pets at the pixel level.")
print("Not perfect after 5 epochs, but the structure is clearly there.")

### 2.5 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
seg_epochs_range = range(1, SEG_EPOCHS + 1)

axes[0].plot(seg_epochs_range, seg_train_losses, 'o-', label='Train', color='#7C6AEF', linewidth=2)
axes[0].plot(seg_epochs_range, seg_test_losses, 's-', label='Test', color='#FB923C', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Segmentation Loss', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(seg_epochs_range, seg_dices, 's-', color='#4ADE80', linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Dice Score')
axes[1].set_title('Test Dice Score', fontweight='bold')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# Recap

### 1. Transfer Learning = Reuse + Adapt
```python
model = models.efficientnet_b0(weights=weights)     # Load pretrained
for p in model.features.parameters():               # Freeze features
    p.requires_grad = False
model.classifier = nn.Linear(1280, num_classes)     # Replace head
```
Leverages millions of images you never labelled. Fast training, high accuracy on small datasets.

### 2. U-Net = Encoder + Decoder + Skip Connections
- **Encoder**: Conv → Pool (compress, learn "what") — same as week 3
- **Decoder**: ConvTranspose2d (expand, recover "where") — NEW
- **Skip connections**: Concatenate encoder → decoder (preserve fine detail)
- **Output**: Per-pixel mask, not a single label

### 3. Same Training Loop — Always
Forward → Loss → Zero Grad → Backward → Step. Week 1, 2, 3, 4 — never changed.

---
*Questions?*